# Q1 - Decision tree baseline and bagging from scratch (Olivetti faces)
Run top to bottom. Only edit `ROLL_NO` in the first code cell. Everything is saved to `split/`, `logs/`, `figures/`.

In [ ]:
# ---- CONFIG ----
ROLL_NO = "0202"                      # your roll number -> seed = 202 (leading zero dropped by int())
SEED = int("".join(ch for ch in ROLL_NO if ch.isdigit()))   # all numeric characters, in order
assert any(ch.isdigit() for ch in ROLL_NO), "Set ROLL_NO first"
print("SEED =", SEED)

DEPTHS = [2, 4, 8, 16, None]
MIN_LEAFS = [1, 2, 4]
B_LIST = [1, 5, 10, 25, 50, 100, 200]
B_MAX = max(B_LIST)
N_CLASSES = 40

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import fetch_olivetti_faces
from sklearn.tree import DecisionTreeClassifier

for d in ["data", "split", "logs", "figures"]:
    os.makedirs(d, exist_ok=True)
print("numpy", np.__version__, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)

faces = fetch_olivetti_faces(data_home="data/sklearn_data", shuffle=False)   # shuffle=False: 10 consecutive images per identity
X = faces.data.astype(np.float64)       # (400, 4096) raw flattened pixels, no PCA / no scaling
y = faces.target.astype(int)
assert X.shape == (400, 4096) and np.bincount(y).tolist() == [10] * 40

In [ ]:
# ---- Split: per identity 6 train / 2 val / 2 test, saved once and re-used everywhere ----
SPLIT_FILE = "split/split_indices.npz"
if os.path.exists(SPLIT_FILE):
    s = np.load(SPLIT_FILE); tr_idx, va_idx, te_idx = s["train"], s["val"], s["test"]
    print("Loaded existing split")
else:
    rng = np.random.default_rng(SEED)
    tr, va, te = [], [], []
    for k in range(N_CLASSES):
        p = rng.permutation(np.where(y == k)[0])
        tr += list(p[:6]); va += list(p[6:8]); te += list(p[8:])
    tr_idx, va_idx, te_idx = np.array(tr), np.array(va), np.array(te)
    np.savez(SPLIT_FILE, train=tr_idx, val=va_idx, test=te_idx, seed=SEED)
    json.dump({"seed": SEED, "train": tr_idx.tolist(), "val": va_idx.tolist(), "test": te_idx.tolist()},
              open("split/split_indices.json", "w"))
    print("Created and saved split")

assert len(set(tr_idx) | set(va_idx) | set(te_idx)) == 400
for part, n in [(tr_idx, 6), (va_idx, 2), (te_idx, 2)]:
    assert (np.bincount(y[part], minlength=40) == n).all()

def mask_images(A):
    B = A.copy().reshape(-1, 64, 64)
    B[:, 24:40, 24:40] = 0.0            # centred 16x16 block, rows/cols 24..39 inclusive
    return B.reshape(len(A), -1)

Xtr, ytr = X[tr_idx], y[tr_idx]
Xva, yva = X[va_idx], y[va_idx]
Xte, yte = X[te_idx], y[te_idx]
Xte_m = mask_images(Xte)                # masked test set (models are trained on clean images only)
print(Xtr.shape, Xva.shape, Xte.shape, Xte_m.shape)

## Data viewing (optional)

In [ ]:
print("X:", X.shape, X.dtype, "| pixel range:", X.min(), X.max())
print("train/val/test:", len(tr_idx), len(va_idx), len(te_idx))
fig, ax = plt.subplots(3, 10, figsize=(14, 4.6))
for r, k in enumerate([0, 1, 2]):
    for c, i in enumerate(np.where(y == k)[0]):
        part = "train" if i in tr_idx else "val" if i in va_idx else "test"
        ax[r, c].imshow(X[i].reshape(64, 64), cmap="gray"); ax[r, c].set_title(f"id{k} {part}", fontsize=7); ax[r, c].axis("off")
plt.tight_layout(); plt.savefig("figures/data_overview_split.pdf"); plt.show()
fig, ax = plt.subplots(2, 6, figsize=(12, 4.5))
for c in range(6):
    ax[0, c].imshow(Xte[c].reshape(64, 64), cmap="gray"); ax[0, c].set_title(f"clean id {yte[c]}", fontsize=8); ax[0, c].axis("off")
    ax[1, c].imshow(Xte_m[c].reshape(64, 64), cmap="gray"); ax[1, c].set_title("masked", fontsize=8); ax[1, c].axis("off")
plt.tight_layout(); plt.savefig("figures/data_overview_mask.pdf"); plt.show()

## Q1(a) Single tree baseline

In [ ]:
def leaf_stats(clf):
    t = clf.tree_
    leaf = t.children_left == -1
    ns = t.n_node_samples[leaf]            # number of TRAINING samples in each leaf
    return int(leaf.sum()), float(ns.mean()), float((ns == 1).mean())

rows, order = [], 0
for d in DEPTHS:
    for m in MIN_LEAFS:
        clf = DecisionTreeClassifier(max_depth=d, min_samples_leaf=m, random_state=SEED).fit(Xtr, ytr)
        nl, msl, f1 = leaf_stats(clf)
        rows.append(dict(order=order, max_depth=str(d), min_samples_leaf=m,
                         train_acc=clf.score(Xtr, ytr), val_acc=clf.score(Xva, yva), test_acc=clf.score(Xte, yte),
                         actual_depth=clf.get_depth(), n_leaves=nl, mean_samples_per_leaf=msl, frac_singleton_leaves=f1))
        order += 1
grid = pd.DataFrame(rows)
grid.to_csv("logs/q1a_grid.csv", index=False)

# Selection uses VALIDATION accuracy only. Ties -> fewer leaves -> earlier in grid order.
best = grid.sort_values(["val_acc", "n_leaves", "order"], ascending=[False, True, True]).iloc[0]
BEST_DEPTH = None if best.max_depth == "None" else int(best.max_depth)
BEST_MSL = int(best.min_samples_leaf)
TREE_PARAMS = dict(max_depth=BEST_DEPTH, min_samples_leaf=BEST_MSL)
json.dump(TREE_PARAMS, open("logs/frozen_tree_params.json", "w"))
print("FROZEN:", TREE_PARAMS)
display(grid.drop(columns="order").round(4))
display(best.to_frame().T.drop(columns="order"))

In [ ]:
# Figure: effect of depth (for each min_samples_leaf)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for m in MIN_LEAFS:
    g = grid[grid.min_samples_leaf == m]
    ax[0].plot(g.max_depth, g.train_acc, "o--", label=f"train msl={m}")
    ax[0].plot(g.max_depth, g.val_acc, "s-", label=f"val msl={m}")
    ax[1].plot(g.max_depth, g.n_leaves, "o-", label=f"msl={m}")
    ax[2].plot(g.max_depth, g.frac_singleton_leaves, "o-", label=f"msl={m}")
ax[0].set_ylabel("accuracy"); ax[1].set_ylabel("#leaves"); ax[2].set_ylabel("frac. singleton leaves")
for a in ax: a.set_xlabel("max_depth"); a.grid(alpha=.3)
ax[0].legend(fontsize=6); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig("figures/q1a_depth.pdf"); plt.show()

In [ ]:
# Frozen single tree: final report numbers
tree = DecisionTreeClassifier(**TREE_PARAMS, random_state=SEED).fit(Xtr, ytr)
nl, msl, f1 = leaf_stats(tree)
q1a = dict(params=TREE_PARAMS, train_acc=tree.score(Xtr, ytr), val_acc=tree.score(Xva, yva), test_acc=tree.score(Xte, yte),
           n_leaves=nl, mean_samples_per_leaf=msl, frac_singleton_leaves=f1)
print(q1a)

## Q1(b),(c) Bagging implemented from scratch
Trees are trained once with B_MAX = 200 bootstrap samples; the ensemble for size B uses the **first B trees** (nested ensembles, so curves are comparable across B).

In [ ]:
class Bagging:
    """Bagging of decision trees. Bootstrap of size n drawn with replacement; majority vote (ties -> smallest label)."""
    def __init__(self, n_estimators, tree_params, seed):
        self.B, self.tree_params, self.seed = n_estimators, tree_params, seed
    def fit(self, X, y):
        rng = np.random.default_rng(self.seed); n = len(X)
        self.trees, self.bags = [], []
        for _ in range(self.B):
            idx = rng.integers(0, n, size=n)
            t = DecisionTreeClassifier(**self.tree_params, random_state=int(rng.integers(2**31 - 1)))
            self.trees.append(t.fit(X[idx], y[idx])); self.bags.append(idx)
        self.inbag = np.zeros((self.B, n), bool)
        for b, idx in enumerate(self.bags): self.inbag[b, idx] = True
        return self
    def tree_preds(self, X):                       # (B, n_samples)
        return np.stack([t.predict(X) for t in self.trees])

def votes_from_preds(P, n_classes=N_CLASSES):      # (n_samples, n_classes) vote counts
    V = np.zeros((P.shape[1], n_classes), int)
    for p in P: V[np.arange(len(p)), p] += 1
    return V

def margin(V):                                     # winning fraction - runner-up fraction
    F = np.sort(V / V.sum(1, keepdims=True), axis=1)
    return F[:, -1] - F[:, -2]

def disagreement(V):
    # mean over samples of the fraction of tree pairs that disagree: (B^2 - sum_k c_k^2) / (B (B-1))
    B = V.sum(1)[0]
    if B < 2: return float("nan")
    return float(((B**2 - (V**2).sum(1)) / (B * (B - 1))).mean())

bag = Bagging(B_MAX, TREE_PARAMS, SEED).fit(Xtr, ytr)
P = {"train": bag.tree_preds(Xtr), "val": bag.tree_preds(Xva), "test": bag.tree_preds(Xte), "test_m": bag.tree_preds(Xte_m)}

def oob_acc(B):
    V = np.zeros((len(ytr), N_CLASSES), int)
    for b in range(B):
        m = ~bag.inbag[b]; V[np.where(m)[0], P["train"][b][m]] += 1
    has = V.sum(1) > 0
    return float((V[has].argmax(1) == ytr[has]).mean()), float(has.mean())

n = len(ytr)
theory_distinct = n * (1 - (1 - 1 / n) ** n)
rows = []
for B in B_LIST:
    Vtr, Vva, Vte = (votes_from_preds(P[k][:B]) for k in ["train", "val", "test"])
    oa, cov = oob_acc(B)
    rows.append(dict(B=B,
        train_acc=(Vtr.argmax(1) == ytr).mean(), val_acc=(Vva.argmax(1) == yva).mean(), test_acc=(Vte.argmax(1) == yte).mean(),
        avg_distinct=np.mean([len(np.unique(i)) for i in bag.bags[:B]]), theory_distinct=theory_distinct,
        oob_acc=oa, oob_coverage=cov,
        disagreement_val=disagreement(Vva), margin_val=margin(Vva).mean()))
res = pd.DataFrame(rows)
res.to_csv("logs/q1bc_bagging.csv", index=False)
print("theory: n(1-(1-1/n)^n) =", theory_distinct, "| limit (1-1/e)n =", (1 - np.exp(-1)) * n)
display(res.round(4))

# Final B: highest VALIDATION accuracy; ties -> smallest B
BEST_B = int(res.sort_values(["val_acc", "B"], ascending=[False, True]).iloc[0].B)
print("FINAL B =", BEST_B)
json.dump({"final_B": BEST_B}, open("logs/final_B.json", "w"))

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
ax[0].plot(res.B, res.val_acc, "o-", label="val"); ax[0].plot(res.B, res.oob_acc, "s-", label="OOB"); ax[0].set_ylabel("accuracy"); ax[0].legend()
ax[1].plot(res.B, res.oob_acc, "s-", color="C1"); ax[1].set_ylabel("OOB accuracy")
ax[2].plot(res.B, res.disagreement_val, "o-", color="C2"); ax[2].set_ylabel("avg pairwise disagreement (val)")
ax[3].plot(res.B, res.margin_val, "o-", color="C3"); ax[3].set_ylabel("mean vote margin (val)")
for a in ax: a.set_xscale("log"); a.set_xlabel("B"); a.grid(alpha=.3); a.axvline(BEST_B, ls=":", c="k")
plt.tight_layout(); plt.savefig("figures/q1c_oob_disagree_margin.pdf"); plt.show()

## Q1(d) Frozen tree vs final bagged model: clean vs masked test

In [ ]:
Vfin = {k: votes_from_preds(P[k][:BEST_B]) for k in ["test", "test_m"]}
sets = {"clean": (Xte, "test"), "masked": (Xte_m, "test_m")}
out, rec = [], {}
for name, (Xs, key) in sets.items():
    tp = tree.predict(Xs); tconf = tree.predict_proba(Xs).max(1)
    V = Vfin[key]; bp = V.argmax(1); bconf = V.max(1) / BEST_B
    rec[name] = dict(tp=tp, bp=bp, V=V, tconf=tconf, bconf=bconf)
    for mdl, p, c in [("single tree", tp, tconf), ("bagging B=%d" % BEST_B, bp, bconf)]:
        out.append(dict(test_set=name, model=mdl, accuracy=(p == yte).mean(), mean_conf=c.mean(),
                        conf_when_correct=c[p == yte].mean() if (p == yte).any() else np.nan,
                        conf_when_wrong=c[p != yte].mean() if (p != yte).any() else np.nan))
comp = pd.DataFrame(out); comp.to_csv("logs/q1d_clean_vs_masked.csv", index=False); display(comp.round(4))

In [ ]:
# Case study: bagging corrects tree (tree wrong, bag right) and both wrong. First two (lowest test index) per set.
case_rows, cases = [], {}
for name in sets:
    r = rec[name]
    fix = np.where((r["tp"] != yte) & (r["bp"] == yte))[0][:2]
    bw = np.where((r["tp"] != yte) & (r["bp"] != yte))[0][:2]
    cases[name] = [("bagging corrects", i) for i in fix] + [("both wrong", i) for i in bw]
    for kind, i in cases[name]:
        V = r["V"][i]; top = np.argsort(-V)[:3]
        case_rows.append(dict(test_set=name, case=kind, test_pos=int(i), dataset_idx=int(te_idx[i]), true=int(yte[i]),
            tree_pred=int(r["tp"][i]), bag_pred=int(r["bp"][i]),
            bag_top3_votes={int(k): int(V[k]) for k in top}, votes_for_true=int(V[yte[i]]), tree_conf=float(r["tconf"][i]), bag_conf=float(r["bconf"][i])))
cases_df = pd.DataFrame(case_rows); cases_df.to_csv("logs/q1d_cases.csv", index=False); display(cases_df)

def train_example(c): return Xtr[np.where(ytr == c)[0][0]].reshape(64, 64)
for name, (Xs, key) in sets.items():
    cs = cases[name]
    if not cs: continue
    fig, ax = plt.subplots(len(cs), 4, figsize=(7, 1.9 * len(cs)), squeeze=False)
    for ri, (kind, i) in enumerate(cs):
        r = rec[name]
        ims = [(Xs[i].reshape(64, 64), f"{kind}\ntrue={yte[i]}"), (train_example(r["tp"][i]), f"tree pred={r['tp'][i]}"),
               (train_example(r["bp"][i]), f"bag pred={r['bp'][i]}"), (train_example(yte[i]), f"true class ex.\nvotes={r['V'][i][yte[i]]}/{BEST_B}")]
        for ci, (im, t) in enumerate(ims):
            ax[ri, ci].imshow(im, cmap="gray"); ax[ri, ci].set_title(t, fontsize=6); ax[ri, ci].axis("off")
    plt.suptitle(f"{name} test set", fontsize=8); plt.tight_layout(); plt.savefig(f"figures/q1d_cases_{name}.pdf"); plt.show()

In [ ]:
# Save final log of all Q1 numbers
final = dict(roll_no=ROLL_NO, seed=SEED, frozen_tree=q1a, final_B=BEST_B,
             bagging_table=res.to_dict(orient="records"), clean_vs_masked=comp.to_dict(orient="records"))
json.dump(final, open("logs/q1_final_log.json", "w"), indent=2, default=str)
print("saved logs/q1_final_log.json")